# survival.ipynb
Survival analyses on the pedpancan cohort.

## Requirements
- `r-survival.yml` environment
- forestploter (installed separately by `survival-plots.R`)

## Changelog
2026-08-06
- Stratified on molecular subtype (186 strata, 26 informative).
- Fitted mixed effects Cox model (`coxme`) on ecDNA x molecular subtype. Slight but nonsignificant variation in estimated effects of molecular tumor type on hazard of ecDNA. Greater for MBL.SHH, MBL.G3; less for HGG.DMG.H3K27.{TP53}; nonsignificant for all.
- Migrate from ggforest -> forestploter (ggforest can't handle mixed effects, stratification, or landmarks).

2025-12-04
- Change *n* threshold to 5; no effect on statistics but we'd like to include ETMR.

2025-11-05  
- remove unused `tumor_types` arg from `load_survival_data`
- add Chapman et al 2023 survival data at runtime
- bugfix Archer samples missing
- bugfix ICGC samples age_at_diagnosis in years not days
- bugfix use z-scores for age
- refactor data imports into separate file

In [ ]:
Sys.setenv(LANGUAGE = "en")

library(tidyverse)
library(readxl)
library(dplyr)
library(stringr)
library(naniar) #for replace with Nas function
library(survival)
library(coxme)
library(survminer)
library(RColorBrewer)
library(janitor)
library(gt)
library(gtsummary)
library(ggsurvfit)
library(extrafont)
library(svglite)

extrafont::font_import(pattern="Arial",prompt=FALSE)
extrafont::loadfonts()

# imports from external file
imports <- new.env()
source("survival-data-imports.R", local = imports)
plotting <- new.env()
source("survival-plots.R",local=plotting)

sessionInfo()

In [ ]:
## create an output directory if it doesn't exist
dir.create('out', showWarnings = FALSE)
## Global path variables
ST_PATH="../../data/Supplementary Tables.xlsx"
CHAPMAN2023_PATH='../../data/external/Chapman2023/41588_2023_1551_MOESM4_ESM.xlsx'

# Kaplan-Meier regression

In [ ]:
# KM amplicon type
# KM by ecDNA status of tumor types with at least 1 sample with ecDNA, censored at 5 years
#dd2 <- load_survival_data("../data/Supplementary Tables 12_1_24.xlsx") %>%
dd2 <- imports$load_survival_data(ST_PATH,CHAPMAN2023_PATH) %>%
  group_by(cancer_type) %>%
  filter(n() >= 5) %>%
  filter (any(OS_status == 'Deceased'))%>%
  filter(any(amplicon_class == 'ecDNA'))%>%
  ungroup()
dd2$cancer_type <- droplevels(dd2$cancer_type) # drop unused levels
dd2$cancer_type %>% unique() # print remaining levels

formula = Surv(OS_months_5y, OS_status_5y) ~ amplicon_class
km = survfit2(formula=formula, data = dd2 )
plotting$km_plot(km)
plotting$save_ggplot("km_class_subset_5year")
logrank <- pairwise_survdiff(formula,dd2,p.adjust.method="BH",rho=0)
logrank


# Cox regressions

We include tumor types which satisfy the following:
- At least 5 patients
- At least one death
- At least one ecDNA

Notes:
- ecDNA is set as the reference level to generate HR relative to chromosomal and no amplification. The values reported in the paper are w.r.t. the other class and may be calculated as 1/HR.
- One can alternately introduce an 'amplified' binary variable comprising the 'ecDNA' and 'chromosomal' classes (see 3rd cell). Since the information contained is identical, so are the resulting regression models.

In [ ]:
# per tumor type: events present AND ecDNA varies
library(dplyr)
dd3 <- imports$load_survival_data(ST_PATH,CHAPMAN2023_PATH)
dd3$molecular_subtype <- with(dd3, ifelse(
  is.na(cancer_subclass),
  as.character(cancer_type),                          # no subtype: stratify on type alone
  paste(cancer_type, cancer_subclass, sep = ".")))            # has subtype: type x subtype
dd3$molecular_subtype <- factor(dd3$molecular_subtype)
# numeric 0/1 versions for the mixed-effects models (coxme random slopes need numerics)
dd3$ecDNA_num <- as.integer(dd3$ecDNA_status == "ecDNA+")
dd3$amp_num   <- as.integer(dd3$amp_status   == "amp.")
dd3 %>% head

## Stratified and mixed-effects models

In [ ]:
# stratify by tumor type, including all tumor types.
m4 <- coxph(Surv(OS_months_5y, OS_status_5y) ~ amp_status + ecDNA_status + strata(cancer_type) + sex + age_at_diagnosis, 
            data = dd3)
#plotting$cox_plot(m4,dd3,"cox_forest_stratified",width=6,height=6)

p <- plotting$forest_with_strata(m4, var_labels = c(
  ecDNA_status        = "ecDNA status",
  amp_status          = "Amplified",
  sex              = "Sex",
  age_at_diagnosis = "Age at diagnosis"))
print(p)

plotting$save_forestploter(p,'out/cox_forest_strat_types.png')
plotting$save_forestploter(p,'out/cox_forest_strat_types.svg')

In [ ]:
# stratify by tumor type, including all tumor types.
m5 <- coxph(Surv(OS_months_5y, OS_status_5y) ~ amp_status + ecDNA_status + strata(molecular_subtype) + sex + age_at_diagnosis, 
            data = dd3)
#plotting$cox_plot(m4,dd3,"cox_forest_stratified",width=6,height=6)

p <- plotting$forest_with_strata(m5, 
  var_labels = c(
    ecDNA_status        = "ecDNA status",
    amp_status          = "Amplified",
    sex              = "Sex",
    age_at_diagnosis = "Age at diagnosis"),
  strata_label = "Molecular subtype"
)
print(p)

plotting$save_forestploter(p,'out/cox_forest_strat_subtypes.png')
plotting$save_forestploter(p,'out/cox_forest_strat_subtypes.svg')

In [ ]:
# Is effect of amplification heterogeneous between molecular subtypes?
# No; \tau^2=0.00011, essentially zero variance for estimates of HR(amp) between subtypes. 
# Indicates confounding in cancer_type estimate that molecular subtype took care of?
m6 <- coxme(Surv(OS_months_5y, OS_status_5y) ~ amp_num + ecDNA_num + 
            strata(molecular_subtype) + (0 + amp_num | molecular_subtype) + sex + age_at_diagnosis, 
            data = dd3)
m6

In [ ]:
m7 <- coxme(Surv(OS_months_5y, OS_status_5y) ~ amp_num + ecDNA_num + 
            strata(molecular_subtype) + (0 + ecDNA_num | molecular_subtype) + sex + age_at_diagnosis, 
            data = dd3)
m7

## Shrinkage caterpillar: per-subtype ecDNA effect

Per molecular subtype, the raw `coxph` MLE of the ecDNA log-HR (single interaction model on the full cohort, `ecDNA_num:molecular_subtype`, the unshrunk analog of `m7`'s random slope) vs the shrunk `coxme` MAP (BLUP from `m7`). MAP intervals use the empirical-Bayes posterior variance. Only the ecDNA-informative subtypes (events on both ecDNA sides) are shown; perfectly-separated or aliased subtypes appear MAP-only ("unstable"/"not estimable").

In [ ]:
# Shrinkage caterpillar inputs for ecDNA: raw coxph MLE (interaction model on the full
# cohort, the unshrunk analog of m7's ecDNA_num random slope) vs shrunk coxme MAP (m7 BLUPs).
# caterpillar_table() pulls the per-subtype MLE + SE from the interaction model and the
# pooled effect / tau^2 / BLUPs from m7, computes the empirical-Bayes MAP CIs, keeps the
# ecDNA-informative subtypes, and attaches the pooled effect as attr(., "pooled").
m_int_ecDNA <- coxph(Surv(OS_months_5y, OS_status_5y) ~ ecDNA_num:molecular_subtype +
                     amp_status + sex + age_at_diagnosis + strata(molecular_subtype), data = dd3)
cat_ecDNA <- plotting$caterpillar_table(m7, m_int_ecDNA, dd3, focus = "ecDNA_num")
#cat_ecDNA
p <- plotting$caterpillar_forest(cat_ecDNA)
print(p)
plotting$save_forestploter(p, 'out/cox_caterpillar_ecDNA.png')
plotting$save_forestploter(p, 'out/cox_caterpillar_ecDNA.svg')

## Shrinkage caterpillar: per-subtype amplification effect

The same MLE-vs-MAP comparison for amplification of any kind (`amp_num`), built from `m6` (random `amp_num` slope). The between-subtype variance of the amplification effect is essentially zero (tau ~ 0.01), so every subtype's noisy MLE is shrunk almost entirely onto the pooled effect: the mixed model finds no subtype-specific amplification signal, in contrast to ecDNA.

In [ ]:
# Amplification caterpillar: same construction from m6 (random amp_num slope). The
# interaction MLE per subtype is the chromosomal-vs-nonamp effect (ecDNA_status held fixed).
m_int_amp <- coxph(Surv(OS_months_5y, OS_status_5y) ~ amp_num:molecular_subtype +
                   ecDNA_status + sex + age_at_diagnosis + strata(molecular_subtype), data = dd3)
cat_amp <- plotting$caterpillar_table(m6, m_int_amp, dd3, focus = "amp_num")
#cat_amp
p <- plotting$caterpillar_forest(cat_amp)
print(p)
plotting$save_forestploter(p, 'out/cox_caterpillar_amp.png')
plotting$save_forestploter(p, 'out/cox_caterpillar_amp.svg')

### Between-subtype heterogeneity (boundary LR tests)

Whether the per-subtype effects genuinely vary is a test of the random-slope variance (H0: `tau^2 = 0`), not something to read off the shrunken MAP intervals (which are conditional on the plug-in `tau_hat` and understate uncertainty, especially when `tau_hat` sits at the zero boundary). Neither effect is significant: ecDNA only hints at heterogeneity (`tau ~ 0.26`, `p ~ 0.16`) and amplification shows none (`tau ~ 0.01`, `p ~ 0.45`), consistent with its MAP estimates collapsing onto the pooled effect.

In [ ]:
# Boundary LR test for between-subtype heterogeneity of each effect (H0: tau^2 = 0).
# Null = the fixed-only model m5; each random-slope model (m7 ecDNA, m6 amp) adds one
# variance component. Because tau^2 is bounded at 0, the reference null is a 50:50
# mixture of chi^2_0 and chi^2_1 (Self & Liang), hence the 0.5 factor.
het_test <- function(mm, null_ll, label) {
  LR <- max(2 * (as.numeric(mm$loglik["Integrated"]) - null_ll), 0)
  p  <- 0.5 * pchisq(LR, df = 1, lower.tail = FALSE)
  cat(sprintf("%-6s heterogeneity: tau = %.3f,  LR = %.3f,  p = %.3f\n",
              label, sqrt(as.numeric(unlist(VarCorr(mm)))[1]), LR, p))
}
het_test(m7, m5$loglik[2], "ecDNA")   # m7: random ecDNA_num slope
het_test(m6, m5$loglik[2], "amp")     # m6: random amp_num slope

In [ ]:
# 26 informative molecular subtypes.
dd3 %>% group_by(molecular_subtype) %>%
  summarise(events    = sum(OS_status_5y),
            amp_var   = n_distinct(amp_status) == 2,
            ev_amp1   = sum(OS_status_5y==1 & amp_status=="amp."),
            ev_amp0   = sum(OS_status_5y==1 & amp_status=="nonamp."),
            informative = ev_amp1 > 0 & ev_amp0 > 0) %>%
  summarise(n_informative = sum(informative),
            events_in_informative = sum(events[informative]))

In [ ]:
dd3 %>% group_by(molecular_subtype) %>%
  summarise(
    ev_chr_only = sum(OS_status_5y==1 & amp_status=="amp." & !(ecDNA_status=="ecDNA+")),
    ev_none     = sum(OS_status_5y==1 & amp_status=="nonamp."),
    informative_for_amp = ev_chr_only > 0 & ev_none > 0,
    .groups="drop") %>%
  summarise(n_informative_amp = sum(informative_for_amp),
            events_amp = sum((ev_chr_only+ev_none)[informative_for_amp]))

## fully parameterized Cox models

In [ ]:
# Set up dataset
dd3 <- imports$load_survival_data(ST_PATH,CHAPMAN2023_PATH) %>%
  group_by(cancer_type) %>%
  filter(any(amplicon_class == 'ecDNA')) %>%
  filter (any(OS_status == 'Deceased'))%>%
  filter(n() >= 5) %>%
  ungroup()
dd3$amplicon_class = relevel(dd3$amplicon_class, ref = "no amplification")
dd3$cancer_type = relevel(dd3$cancer_type, ref = "LGG")
dd3$cancer_type %>% unique()
dd3$cancer_type <- droplevels(dd3$cancer_type)

# print summary
dd3 %>% group_by(amplicon_class, cancer_type) %>%
  summarise(n=n())%>%
  spread(cancer_type, n)

# cox regression
m4 <- coxph(Surv(OS_months_5y, OS_status_5y) ~ amplicon_class + cancer_type + sex + age_at_diagnosis, data = dd3)
plotting$cox_plot(m4, dd3)
plotting$save_ggplot("cox_forest_v2", width = 6, height = 6)

In [ ]:
# HRs are easier to calculate when the reference is ecDNA
dd3$amplicon_class = relevel(dd3$amplicon_class, ref = "ecDNA")
# cox regression
m4 <- coxph(Surv(OS_months_5y, OS_status_5y) ~ amplicon_class + cancer_type + sex + age_at_diagnosis, data = dd3)
print(paste("Hazard ratio ecDNA vs chromosomal:",round(1/exp(m4$coefficients[['amplicon_classchromosomal']]),2),"; p=",(summary(m4)$coefficients[['amplicon_classchromosomal','Pr(>|z|)']])))
print(paste("Hazard ratio ecDNA vs no amplification:",round(1/exp(m4$coefficients[['amplicon_classno amplification']]),2),"; p=",(summary(m4)$coefficients[['amplicon_classno amplification','Pr(>|z|)']])))

In [ ]:
# Set up dataset
#dd3 <-load_survival_data("../data/Supplementary Tables 12_1_24.xlsx") %>%
dd3 <- imports$load_survival_data(ST_PATH,CHAPMAN2023_PATH) %>%
group_by(cancer_type) %>%
  filter(any(amplicon_class == 'ecDNA')) %>%
  filter (any(OS_status == 'Deceased'))%>%
  filter(n() >= 5) %>%
  ungroup()
dd3$cancer_type = relevel(dd3$cancer_type, ref = "LGG")
dd3$amp_status = relevel(dd3$amp_status, ref = 1)
dd3$ecDNA_status = relevel(dd3$ecDNA_status, ref = 2)
dim(dd3)
dd3$cancer_type %>% unique()
dd3$cancer_type <- droplevels(dd3$cancer_type)

# print summary
dd3 %>% group_by(amplicon_class, cancer_type) %>%
  summarise(n=n())%>%
  spread(cancer_type, n)

# cox regression
m4 <- coxph(Surv(OS_months_5y, OS_status_5y) ~ ecDNA_status + amp_status + cancer_type + sex + age_at_diagnosis, data = dd3)
m4
plotting$cox_plot(m4, dd3)
plotting$save_ggplot("cox_forest", width = 6, height = 6)

In [ ]:
# check proportional hazards assumptions
cox.zph(m4)

# More plots, not included in the manuscript

## CNS tumor specific regressions

In [ ]:
# Set up dataset
cns_tumor_types = c('ETMR','GNT','LGG','MNG','HGG','EPN','MBL','SEGA','CPG','EMBT','ATRT','CHDM','CPT','PINT','BTNOS')
dd3 <- imports$load_survival_data(ST_PATH,CHAPMAN2023_PATH) %>%
  filter(cancer_type %in% cns_tumor_types) %>%
group_by(cancer_type) %>%
  filter(any(amplicon_class == 'ecDNA')) %>%
  filter (any(OS_status == 'Deceased'))%>%
  filter(n() >= 5) %>%
  ungroup()
dd3$cancer_type = relevel(dd3$cancer_type, ref = "LGG")
dd3$amp_status = relevel(dd3$amp_status, ref = 1)
dd3$ecDNA_status = relevel(dd3$ecDNA_status, ref = 2)
dim(dd3)
dd3$cancer_type %>% unique()
dd3$cancer_type <- droplevels(dd3$cancer_type)

# print summary
dd3 %>% group_by(amplicon_class, cancer_type) %>%
  summarise(n=n())%>%
  spread(cancer_type, n)

# cox regression
m4 <- coxph(Surv(OS_months_5y, OS_status_5y) ~ ecDNA_status + amp_status + cancer_type + sex + age_at_diagnosis, data = dd3)
m4
plotting$cox_plot(m4, dd3)
plotting$save_ggplot("cox_bt_forest", width = 6, height = 6)

In [ ]:
dd3$amplicon_class = relevel(dd3$amplicon_class, ref = "no amplification")
m4 <- coxph(Surv(OS_months_5y, OS_status_5y) ~ amplicon_class + cancer_type + sex + age_at_diagnosis, data = dd3)
plotting$cox_plot(m4, dd3)
plotting$save_ggplot("cox_bt_forest_v2", width = 6, height = 6)

In [ ]:
formula = Surv(OS_months_5y, OS_status_5y) ~ amplicon_class
km = survfit2(formula=formula, data = dd3)
plotting$km_plot(km)
plotting$save_ggplot("km_bt_5year")
logrank <- pairwise_survdiff(formula,dd3,p.adjust.method="BH",rho=0)
logrank

## Alternative KM curves on the whole cohort

In [ ]:
# KM by ecDNA+/- status of combined cohort, censored at 5 years
#no filters applied by cancer type or n value
print(getwd())
data <- imports$load_survival_data(ST_PATH,CHAPMAN2023_PATH)
formula <- Surv(OS_months_5y, OS_status_5y) ~ ecDNA_status
km <- survfit2(formula=formula, data=data)
plotting$km_plot(km)
#plotting$km_plot(km, "km_surv_all_5year")
logrank <- survdiff(formula,data)
logrank
dim(data)

In [ ]:
# KM by amplicon type
# no filters applied by tumor type or n value
data <- imports$load_survival_data(ST_PATH,CHAPMAN2023_PATH)
  #mutate(amplicon_class = recode(amplicon_class))
formula <- Surv(OS_months_5y, OS_status_5y) ~ amplicon_class
km <- survfit2(formula=formula, data=data)
plotting$km_plot(km)
#plotting$km_plot(km, "km_class_all_5year")
logrank <- pairwise_survdiff(formula,data,p.adjust.method="BH",rho=0)
logrank

In [ ]:
# KM by ecDNA+/- status of tumor types with at least 1 sample with ecDNA, censored at 5 years
#filtered by tumor type with ecDNA, n > 10 and at least one patient who is deceased
dd2 <- imports$load_survival_data(ST_PATH,CHAPMAN2023_PATH) %>%
  group_by(cancer_type) %>%
  filter(any(amplicon_class == 'ecDNA')) %>%
  filter(n() >= 10) %>%
  filter (any(OS_status == 'Deceased'))%>%
  ungroup()
dd2$cancer_type <- droplevels(dd2$cancer_type) # drop unused levels
dd2$cancer_type %>% unique() # print remaining levels

formula <- Surv(OS_months_5y, OS_status_5y) ~ ecDNA_status
km <- survfit2(formula=formula, data=dd2)
plotting$km_plot(km)
#plotting$km_plot(km, "km_surv_subset_5year")
logrank <- survdiff(formula,dd2)
logrank
